# VULCAN Emulator — Extras Notebook

Self-contained demos and diagnostics for a trained FastChem emulator bundle.
Sections below are independent; set `MODEL` once, then run any section.

1. **Standalone inference** — simplest inference path (no `src/` imports).
2. **Emulator demo** — run on one saved test profile, compare with the stored target, save a plot.
3. **Saved test P-T profiles** — visualize the test-set T-P grid by source category.
4. **FastChem rerun comparison** — rerun FastChem and overlay on the stored profile.
5. **Chemistry diagnostic** — mean-absolute log10 error across transformer / stored / FastChem.

## Setup

In [ ]:
from __future__ import annotations

import subprocess
import sys
import tempfile
import types
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import h5py
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from matplotlib.lines import Line2D

# Allow running from either extras/ or the project root.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "extras":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_generation.data_loader import ProcessedSplit, load_processed_dataset
from src.data_generation.generation import (
    _copy_fastchem_runtime,
    list_run_ids_from_consolidated,
)
from src.data_generation.preprocess import inverse_block, inverse_mixed_block
from src.models.export_bundle import load_exported_model
from src.utils.helpers import resolve_path

# ----- The single knob for this notebook -----
MODEL = "fastchem_no_condensation"

BUNDLE_PATH = PROJECT_ROOT / "models" / MODEL / "best_exported.npz"
PLOTS_DIR = BUNDLE_PATH.parent / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
STYLE_PATH = PROJECT_ROOT / "extras" / "science.mplstyle"
if STYLE_PATH.exists():
    plt.style.use(str(STYLE_PATH))

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"MODEL        : {MODEL}")
print(f"BUNDLE_PATH  : {BUNDLE_PATH}")

## Shared utilities

Test-split / raw-dataset loading, FastChem rerun support, plot helpers.

In [ ]:
EPSILON = 1.0e-30

SOLAR_ELEMENT_ABUNDANCES = {
    "He_H": 7.84e-2, "C_H": 2.69e-4, "O_H": 4.90e-4,
    "N_H": 6.76e-5, "S_H": 1.32e-5,
}
FASTCHEM_METALLICITY_SCALED_ELEMENTS = {
    "C", "N", "O", "S", "P", "Si", "Ti", "V",
    "Cl", "K", "Na", "Mg", "F", "Ca", "Fe",
}


@dataclass(frozen=True)
class FastChemTestContext:
    bundle_path: Path
    processed_root: Path
    raw_root: Path | None
    split: ProcessedSplit
    normalization: dict[str, Any]
    contract: dict[str, Any]
    run_id_to_index: dict[str, int]
    raw_run_ids: frozenset[str]


@dataclass(frozen=True)
class FastChemTestCase:
    run_id: str
    pressure_bar: np.ndarray
    temperature_k: np.ndarray
    global_inputs: dict[str, float]
    stored_target_ymix: np.ndarray
    output_species: list[str]
    raw_globals: dict[str, float] | None
    raw_metadata: dict[str, Any] | None


@dataclass(frozen=True)
class RawEquilibriumProfile:
    run_id: str
    pressure_bar: np.ndarray
    temperature_k: np.ndarray
    equilibrium_ymix: np.ndarray
    output_species: list[str]
    globals: dict[str, float]


def _decode_labels(values: np.ndarray) -> list[str]:
    return [v.decode("utf-8") if isinstance(v, bytes) else str(v) for v in values]


def _decode_scalar(value: Any) -> Any:
    scalar = np.asarray(value)[()]
    if isinstance(scalar, bytes): return scalar.decode("utf-8")
    if isinstance(scalar, np.bool_): return bool(scalar)
    if isinstance(scalar, np.integer): return int(scalar)
    if isinstance(scalar, np.floating): return float(scalar)
    return scalar


def _resolve_processed_root(bundle_path: Path, config: dict) -> Path:
    paths = config.get("paths", {}) if isinstance(config.get("paths"), dict) else {}
    candidates: list[Path] = []
    processed_value = paths.get("processed_root") if isinstance(paths, dict) else None
    if isinstance(processed_value, str) and processed_value:
        candidates.append(resolve_path(processed_value, PROJECT_ROOT))
    candidates.append(PROJECT_ROOT / "data" / bundle_path.parent.name / "processed")
    for candidate in candidates:
        if (candidate / "test" / "metadata.json").exists():
            return candidate
    raise FileNotFoundError(f"No processed test split under: {candidates}")


def _resolve_raw_root(bundle_path: Path, config: dict, processed_root: Path):
    paths = config.get("paths", {}) if isinstance(config.get("paths"), dict) else {}
    candidates: list[Path] = []
    raw_value = paths.get("raw_root") if isinstance(paths, dict) else None
    if isinstance(raw_value, str) and raw_value:
        candidates.append(resolve_path(raw_value, PROJECT_ROOT))
    candidates.append(processed_root.parent / "raw")
    candidates.append(PROJECT_ROOT / "data" / bundle_path.parent.name / "raw")
    for candidate in candidates:
        consolidated = candidate / "runs.h5"
        if consolidated.exists():
            run_ids = frozenset(list_run_ids_from_consolidated(consolidated))
            if run_ids:
                return candidate, run_ids
    return None, frozenset()


def load_fastchem_test_context(bundle_path: Path, config: dict, *, require_raw: bool = False) -> FastChemTestContext:
    processed_root = _resolve_processed_root(bundle_path, config)
    splits, normalization, contract = load_processed_dataset(processed_root)
    split = splits["test"]
    raw_root, raw_run_ids = _resolve_raw_root(bundle_path, config, processed_root)
    if require_raw and raw_root is None:
        raise FileNotFoundError("Raw dataset required but not found.")
    return FastChemTestContext(
        bundle_path=bundle_path,
        processed_root=processed_root,
        raw_root=raw_root,
        split=split,
        normalization=normalization,
        contract=contract,
        run_id_to_index={rid: i for i, rid in enumerate(split.run_ids)},
        raw_run_ids=raw_run_ids,
    )


def _load_raw_sidecars(raw_root: Path, run_id: str):
    with h5py.File(raw_root / "runs.h5", "r") as handle:
        group = handle[run_id]
        globals_map = {k: float(np.asarray(group[f"globals/{k}"])) for k in group["globals"].keys()}
        if "inputs/element_input_order" in group and "inputs/elemental_abundances_frac" in group:
            labels = _decode_labels(np.asarray(group["inputs/element_input_order"]))
            profile = np.asarray(group["inputs/elemental_abundances_frac"], dtype=np.float64)
            for idx, label in enumerate(labels):
                globals_map[label] = float(profile[0, idx])
        metadata: dict[str, Any] = {}
        if "metadata" in group:
            metadata = {k: _decode_scalar(group[f"metadata/{k}"][()]) for k in group["metadata"].keys()}
    return globals_map, metadata


def load_fastchem_raw_metadata_map(raw_root: Path, run_ids: list[str]) -> dict[str, dict[str, Any]]:
    out: dict[str, dict[str, Any]] = {}
    with h5py.File(raw_root / "runs.h5", "r") as handle:
        for rid in run_ids:
            group = handle[rid]
            if "metadata" not in group:
                out[rid] = {}; continue
            out[rid] = {k: _decode_scalar(group[f"metadata/{k}"][()]) for k in group["metadata"].keys()}
    return out


def classify_temperature_profile_bucket(metadata: dict[str, Any]) -> str:
    source = metadata.get("temperature_profile_source")
    if source == "pt_library":
        return "pt_library"
    if source == "analytic":
        key = "temperature_profile_analytic_convective_adjustment_applied"
        return "analytic_convective" if bool(metadata[key]) else "analytic_radiative"
    raise KeyError("Unsupported temperature_profile_source.")


def load_fastchem_test_case(ctx: FastChemTestContext, run_id: str) -> FastChemTestCase:
    idx = ctx.run_id_to_index[run_id]
    seq_in = np.asarray(ctx.split.sequence_inputs[idx], dtype=np.float64)
    glob_in = np.asarray(ctx.split.global_inputs[idx:idx + 1], dtype=np.float64)
    target = np.asarray(ctx.split.target_outputs[idx], dtype=np.float64)

    pressure_bar = inverse_block(seq_in[:, 0:1], ctx.normalization["sequence_static"]["blocks"][0])[:, 0]
    temperature_k = inverse_block(seq_in[:, 1:2], ctx.normalization["sequence_static"]["blocks"][1])[:, 0]
    global_vec = inverse_mixed_block(glob_in, ctx.normalization["global_static"])[0]
    order = list(ctx.contract["global_static_feature_order"])
    global_map = {n: float(global_vec[i]) for i, n in enumerate(order)}
    stored = inverse_block(target, ctx.normalization["target"])

    raw_globals = raw_metadata = None
    if ctx.raw_root is not None and run_id in ctx.raw_run_ids:
        raw_globals, raw_metadata = _load_raw_sidecars(ctx.raw_root, run_id)

    return FastChemTestCase(
        run_id=run_id,
        pressure_bar=np.asarray(pressure_bar, dtype=np.float64),
        temperature_k=np.asarray(temperature_k, dtype=np.float64),
        global_inputs=global_map,
        stored_target_ymix=np.asarray(stored, dtype=np.float64),
        output_species=list(ctx.contract["output_species_order"]),
        raw_globals=raw_globals,
        raw_metadata=raw_metadata,
    )


def load_raw_equilibrium_profile(raw_root: Path, run_id: str) -> RawEquilibriumProfile:
    with h5py.File(raw_root / "runs.h5", "r") as handle:
        g = handle[run_id]
        pressure_bar = np.asarray(g["inputs/pressure_bar"], dtype=np.float64)
        temperature_k = np.asarray(g["inputs/temperature_k"], dtype=np.float64)
        output_species = _decode_labels(np.asarray(g["inputs/output_species"]))
        equilibrium_ymix = np.asarray(g["equilibrium/ymix"], dtype=np.float64)
        globals_map = {k: float(np.asarray(g[f"globals/{k}"])) for k in g["globals"].keys()}
        if "inputs/element_input_order" in g and "inputs/elemental_abundances_frac" in g:
            labels = _decode_labels(np.asarray(g["inputs/element_input_order"]))
            profile = np.asarray(g["inputs/elemental_abundances_frac"], dtype=np.float64)
            for i, label in enumerate(labels):
                globals_map[label] = float(profile[0, i])
    return RawEquilibriumProfile(
        run_id=run_id, pressure_bar=pressure_bar, temperature_k=temperature_k,
        equilibrium_ymix=equilibrium_ymix, output_species=output_species, globals=globals_map,
    )


def resolve_vulcan_source_root(config: dict) -> Path:
    paths = config.get("paths", {})
    root = paths.get("vulcan_source_root") if isinstance(paths, dict) else None
    if not isinstance(root, str) or not root:
        raise KeyError("Bundle config missing paths.vulcan_source_root.")
    return resolve_path(root, PROJECT_ROOT)


# ----- FastChem online rerun -----

def _sulfur_enabled(config: dict) -> bool:
    spec = config.get("data_spec", {})
    names = list(spec.get("state_species", [])) + list(spec.get("output_species", []))
    return any("S" in n for n in names)


def _explicit_elements(config: dict) -> set[str]:
    atoms = {"O", "C", "N", "He"}
    if _sulfur_enabled(config):
        atoms.add("S")
    return atoms


def _element_abundances(globals_map: dict[str, float]) -> dict[str, float]:
    if "metallicity_log10" in globals_map:
        met_scale = 10.0 ** float(globals_map["metallicity_log10"])
    else:
        met_scale = float(globals_map["O_H"]) / SOLAR_ELEMENT_ABUNDANCES["O_H"]
    return {k: float(globals_map[k]) for k in ("He_H", "C_H", "O_H", "N_H", "S_H")} | {"fastchem_met_scale": met_scale}


def _write_tp(fastchem_root: Path, pressure_bar: np.ndarray, temperature_k: np.ndarray) -> None:
    tp_dir = fastchem_root / "input" / "vulcan_TP"
    tp_dir.mkdir(parents=True, exist_ok=True)
    with (tp_dir / "vulcan_TP.dat").open("w", encoding="utf-8") as fh:
        fh.write("#p (bar)    T (K)\n")
        for p, t in zip(pressure_bar, temperature_k):
            fh.write(f"{p:.3e}\t{t:.1f}\n")


def _write_abundances(fastchem_root: Path, globals_map: dict[str, float], config: dict) -> None:
    input_dir = fastchem_root / "input"
    params = "parameters_ion.dat" if config.get("physics_toggles", {}).get("use_ion_chemistry") else "parameters_wo_ion.dat"
    (input_dir / "parameters.dat").write_text((input_dir / params).read_text(encoding="utf-8"), encoding="utf-8")

    ab = _element_abundances(globals_map)
    met_offset = float(np.log10(ab["fastchem_met_scale"]))
    explicit = _explicit_elements(config)
    solar_file = input_dir / "solar_element_abundances.dat"

    out_lines: list[str] = []
    for line in solar_file.read_text(encoding="utf-8").splitlines(keepends=True):
        if not line.strip() or line.startswith("#"):
            out_lines.append(line); continue
        parts = line.split()
        name = parts[0]
        if name in explicit:
            key = "He_H" if name == "He" else f"{name}_H"
            out_lines.append(f"{name}\t{np.log10(ab[key]) + 12.0:.4f}\n")
        elif name in FASTCHEM_METALLICITY_SCALED_ELEMENTS:
            out_lines.append(f"{name}\t{float(parts[1]) + met_offset:.4f}\n")
        else:
            out_lines.append(line)
    (input_dir / "element_abundances_vulcan.dat").write_text("".join(out_lines), encoding="utf-8")


def _load_fastchem_output(output_path: Path, species: list[str]) -> np.ndarray:
    payload = np.genfromtxt(output_path, names=True, dtype=None, encoding=None)
    rows = np.atleast_1d(payload)
    return np.column_stack([np.asarray(rows[n], dtype=np.float64) for n in species])


def run_fastchem_online(
    source_root: Path, pressure_bar: np.ndarray, temperature_k: np.ndarray,
    globals_map: dict[str, float], output_species: list[str], config: dict,
) -> np.ndarray:
    with tempfile.TemporaryDirectory(prefix="fastchem_compare_") as tmp:
        fastchem_root = _copy_fastchem_runtime(source_root, Path(tmp))
        _write_abundances(fastchem_root, globals_map, config)
        _write_tp(fastchem_root, pressure_bar, temperature_k)
        result = subprocess.run(
            ["./fastchem", "input/config.input"], cwd=fastchem_root, check=False,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(f"FastChem failed:\n{result.stdout}")
        return _load_fastchem_output(fastchem_root / "output" / "vulcan_EQ.dat", output_species)


# Load the bundle once; every section below reuses `model`.
if not BUNDLE_PATH.exists():
    raise FileNotFoundError(f"Bundle not found: {BUNDLE_PATH}")
model = load_exported_model(BUNDLE_PATH)
print(f"Loaded {model.model_type} ({model.chemistry_type}) with {len(model.data_contract['output_species_order'])} species.")

## 1. Standalone inference

Simplest inference path: the bundle carries its own inference module, so no `src/` import is needed.

In [ ]:
_src = bytes(np.load(BUNDLE_PATH, allow_pickle=False)["meta/vulcan_emulator_src"]).decode()
_mod = types.ModuleType("vulcan_emulator_standalone")
sys.modules["vulcan_emulator_standalone"] = _mod
exec(compile(_src, "vulcan_emulator_standalone", "exec"), _mod.__dict__)
standalone_model = _mod.load_model(BUNDLE_PATH)

# 50-level pressure grid matching the training domain: 100 -> 1e-7 bar.
pressure_bar = np.logspace(2, -7, 50)
temperature_k = np.full_like(pressure_bar, 1500.0)

predictions_log10 = np.asarray(
    standalone_model.predict_fastchem(
        pressure_bar=pressure_bar,
        temperature_k=temperature_k,
        global_inputs=SOLAR_ELEMENT_ABUNDANCES,
        return_log10=True,
    )
)

idx = int(np.argmin(np.abs(pressure_bar - 0.1)))
print(f"Output shape: {predictions_log10.shape}")
print("Mixing ratios at P = 0.1 bar (log10):")
for name in ["H2", "H2O", "CO", "CO2", "CH4", "NH3", "H2S"]:
    if name in standalone_model.species:
        col = standalone_model.species.index(name)
        print(f"  {name:>5s} = {predictions_log10[idx, col]:+.3f}")

## 2. Emulator demo

Run the emulator on one saved processed test profile and plot it against the stored target.

In [ ]:
context = load_fastchem_test_context(BUNDLE_PATH, model.config)
test_case = load_fastchem_test_case(context, context.split.run_ids[0])
species = test_case.output_species

emulator_log10 = np.asarray(
    model.predict_fastchem_profile(
        pressure_bar=test_case.pressure_bar,
        temperature_k=test_case.temperature_k,
        global_inputs=test_case.global_inputs,
        return_log10=True,
    )
)
emulator_ymix = np.power(10.0, emulator_log10)
stored_log10 = np.log10(np.clip(test_case.stored_target_ymix, EPSILON, None))
phot_idx = int(np.argmin(np.abs(test_case.pressure_bar - 0.1)))

print(f"Run: {test_case.run_id}  ({test_case.pressure_bar.size} levels)")
print("Mixing ratios at P = 0.1 bar (log10):")
for name in ["H2", "H2O", "CO", "CO2", "CH4", "NH3", "H2S"]:
    if name in species:
        i = species.index(name)
        print(f"  {name:>5s}  stored = {stored_log10[phot_idx, i]:+.3f}, emulator = {emulator_log10[phot_idx, i]:+.3f}")

fig, (ax_pt, ax_mix) = plt.subplots(1, 2, figsize=(14, 7), sharey=True)
colors = plt.cm.tab20(np.linspace(0, 1, len(species)))

ax_pt.plot(test_case.temperature_k, test_case.pressure_bar, color="black", lw=2.0)
ax_pt.set_xlabel("Temperature [K]")
ax_pt.set_ylabel("Pressure [bar]")
ax_pt.set_yscale("log")
ax_pt.invert_yaxis()
ax_pt.set_xlim(0.0, 3000.0)
ax_pt.set_title("Stored Test P-T Profile")
ax_pt.yaxis.set_major_locator(mticker.LogLocator(base=10, numticks=8))
ax_pt.yaxis.set_minor_locator(mticker.NullLocator())

for i, name in enumerate(species):
    ax_mix.plot(np.clip(test_case.stored_target_ymix[:, i], EPSILON, None),
                test_case.pressure_bar, color=colors[i], lw=1.6, label=name)
    ax_mix.plot(np.clip(emulator_ymix[:, i], EPSILON, None),
                test_case.pressure_bar, color=colors[i], lw=1.2, ls="--")
ax_mix.set_xscale("log")
ax_mix.set_xlim(1.0e-20, 3.0)
ax_mix.set_xlabel("Mixing Ratio")
ax_mix.set_title("Stored Test vs Emulator")

species_legend = ax_mix.legend(fontsize=7, ncol=3, loc="lower left")
ax_mix.add_artist(species_legend)
ax_mix.legend(handles=[
    Line2D([0], [0], color="black", lw=1.6, label="Stored test"),
    Line2D([0], [0], color="black", lw=1.2, ls="--", label="Emulator"),
], fontsize=8, loc="upper right")
fig.suptitle(test_case.run_id, fontsize=11)
fig.tight_layout()

output_path = PLOTS_DIR / "emulator_demo.png"
fig.savefig(output_path, dpi=160)
plt.show()
print(f"Saved: {output_path}")

## 3. Saved test P-T profiles

Overlay test-set T-P profiles grouped by source (PT library vs analytic).

In [ ]:
PROFILES_PER_BUCKET = 5
BUCKET_STYLE = {
    "pt_library": {"cmap": "Reds", "ls": "-", "lw": 2.2, "label": "PT-library"},
    "analytic_radiative": {"cmap": "Blues", "ls": "--", "lw": 2.2, "label": "Analytic (radiative)"},
    "analytic_convective": {"cmap": "Purples", "ls": (0, (5, 2, 1, 2)), "lw": 2.5, "label": "Analytic (conv. adj.)"},
}

ctx_raw = load_fastchem_test_context(BUNDLE_PATH, model.config, require_raw=True)
metadata_map = load_fastchem_raw_metadata_map(ctx_raw.raw_root, ctx_raw.split.run_ids)

buckets: dict[str, list[str]] = {n: [] for n in BUCKET_STYLE}
for rid in ctx_raw.split.run_ids:
    buckets[classify_temperature_profile_bucket(metadata_map[rid])].append(rid)

fig, ax = plt.subplots(figsize=(8, 8))
for name, rids in buckets.items():
    style = BUCKET_STYLE[name]
    cases = [load_fastchem_test_case(ctx_raw, rid) for rid in rids[:PROFILES_PER_BUCKET]]
    cmap = plt.get_cmap(style["cmap"])
    palette = [cmap(0.35 + 0.55 * i / max(len(cases) - 1, 1)) for i in range(len(cases))]
    for i, case in enumerate(cases):
        ax.plot(case.temperature_k, case.pressure_bar,
                ls=style["ls"], lw=style["lw"], alpha=0.8, color=palette[i],
                label=style["label"] if i == 0 else None)

ax.set_yscale("log")
ax.set_ylim(1.0e2, 1.0e-5)
ax.set_xlim(0.0, 3000.0)
ax.set_xlabel("Temperature [K]")
ax.set_ylabel("Pressure [bar]")
ax.set_title("Saved Test P-T Profiles")
ax.legend(loc="best")

output_path = PLOTS_DIR / "saved_test_profiles.png"
fig.savefig(output_path)
plt.show()
print(f"Saved: {output_path}")

## 4. FastChem rerun comparison

Pick a raw test profile, rerun FastChem with identical inputs, and overlay.

In [ ]:
ctx_raw = load_fastchem_test_context(BUNDLE_PATH, model.config, require_raw=True)
run_id = next(rid for rid in ctx_raw.split.run_ids if rid in ctx_raw.raw_run_ids)
profile = load_raw_equilibrium_profile(ctx_raw.raw_root, run_id)

fastchem_ymix = run_fastchem_online(
    source_root=resolve_vulcan_source_root(model.config),
    pressure_bar=profile.pressure_bar,
    temperature_k=profile.temperature_k,
    globals_map=profile.globals,
    output_species=profile.output_species,
    config=model.config,
)

fig, (ax_pt, ax_mix, ax_delta) = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
colors = plt.cm.tab20(np.linspace(0, 1, len(profile.output_species)))
truth = np.clip(profile.equilibrium_ymix, EPSILON, None)
rerun = np.clip(fastchem_ymix, EPSILON, None)

ax_pt.plot(profile.temperature_k, profile.pressure_bar, color="black", lw=2.0)
ax_pt.set_xlabel("Temperature [K]")
ax_pt.set_ylabel("Pressure [bar]")
ax_pt.set_yscale("log")
ax_pt.invert_yaxis()
ax_pt.set_xlim(0.0, 3000.0)
ax_pt.set_title("Test P-T Profile")

for i, name in enumerate(profile.output_species):
    ax_mix.plot(truth[:, i], profile.pressure_bar, color=colors[i], lw=1.6, label=name)
    ax_mix.plot(rerun[:, i], profile.pressure_bar, color=colors[i], lw=1.2, ls="--")
    ax_delta.plot(np.log10(rerun[:, i] + EPSILON) - np.log10(truth[:, i] + EPSILON),
                  profile.pressure_bar, color=colors[i], lw=1.3)

ax_mix.set_xscale("log")
ax_mix.set_xlim(1.0e-20, 3.0)
ax_mix.set_xlabel("Mixing Ratio")
ax_mix.set_title("Mixing Ratios")
species_legend = ax_mix.legend(fontsize=7, ncol=3, loc="lower left")
ax_mix.add_artist(species_legend)
ax_mix.legend(handles=[
    Line2D([0], [0], color="black", lw=1.6, label="Test"),
    Line2D([0], [0], color="black", lw=1.2, ls="--", label="FastChem"),
], fontsize=8, loc="upper left")

ax_delta.axvline(0.0, color="black", lw=1.0, alpha=0.6)
ax_delta.set_xlim(-0.5, 0.5)
ax_delta.set_xlabel(r"$\log_{10}(\mathrm{FastChem}) - \log_{10}(\mathrm{Test})$")
ax_delta.set_title("FastChem Residual")

fig.suptitle(profile.run_id, fontsize=12)
fig.tight_layout()
output_path = PLOTS_DIR / f"{profile.run_id}_fastchem_compare.png"
fig.savefig(output_path, dpi=180)
plt.show()
print(f"Saved: {output_path}")

## 5. Chemistry diagnostic

Mean-absolute log10 error: transformer vs stored target vs live FastChem.

In [ ]:
def _mean_abs_log10_error(left: np.ndarray, right: np.ndarray) -> float:
    left_c = np.clip(left, EPSILON, None)
    right_c = np.clip(right, EPSILON, None)
    return float(np.mean(np.abs(np.log10(left_c) - np.log10(right_c))))


ctx_raw = load_fastchem_test_context(BUNDLE_PATH, model.config, require_raw=True)
run_id = next(rid for rid in ctx_raw.split.run_ids if rid in ctx_raw.raw_run_ids)
test_case = load_fastchem_test_case(ctx_raw, run_id)
species = test_case.output_species

transformer_vmr = np.asarray(
    model.predict_fastchem_profile(
        pressure_bar=test_case.pressure_bar,
        temperature_k=test_case.temperature_k,
        global_inputs=test_case.global_inputs,
        return_log10=False,
    )
)
fastchem_vmr = run_fastchem_online(
    source_root=resolve_vulcan_source_root(model.config),
    pressure_bar=test_case.pressure_bar,
    temperature_k=test_case.temperature_k,
    globals_map=test_case.raw_globals or test_case.global_inputs,
    output_species=species,
    config=model.config,
)
stored_vmr = test_case.stored_target_ymix
phot_idx = int(np.argmin(np.abs(test_case.pressure_bar - 0.1)))

print(f"Run: {test_case.run_id}  ({test_case.pressure_bar.size} levels)\n")
print("Mean |delta log10 VMR|:")
print(f"  transformer vs stored   = {_mean_abs_log10_error(transformer_vmr, stored_vmr):.4f}")
print(f"  transformer vs FastChem = {_mean_abs_log10_error(transformer_vmr, fastchem_vmr):.4f}")
print(f"  stored vs FastChem      = {_mean_abs_log10_error(stored_vmr, fastchem_vmr):.4f}")

print(f"\nVMR at P = 0.1 bar:")
print(f"  {'species':>6s}  {'transformer':>12s}  {'stored':>12s}  {'fastchem':>12s}")
for name in ("H2", "He", "CO", "H2O", "CH4", "NH3", "H2S"):
    if name not in species:
        continue
    i = species.index(name)
    print(
        f"  {name:>6s}  {transformer_vmr[phot_idx, i]:12.4e}  "
        f"{stored_vmr[phot_idx, i]:12.4e}  {fastchem_vmr[phot_idx, i]:12.4e}"
    )